# 05 — Evaluation & Final Results

Comprehensive evaluation of all models:
- Supervised Siamese U-Net
- Self-Supervised Autoencoder
- NDVI Difference Baseline

Generates publication-quality figures for the README.

In [ ]:
!pip install torchgeo matplotlib numpy seaborn -q

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tempfile
import torch
from torch.utils.data import DataLoader
from torchgeo.datasets import OSCD100
from tqdm.auto import tqdm

if os.path.exists('delta/src'):
    sys.path.insert(0, 'delta')
elif os.path.exists('../src'):
    sys.path.insert(0, '..')

from src.models.siamese_unet import SiameseUNet
from src.models.unet import Autoencoder
from src.training.metrics import evaluate_all, compute_f1, compute_iou
from src.data.ndvi import compute_ndvi_torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Load Models

In [ ]:
def load_supervised(path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = SiameseUNet(
        in_channels=4, out_channels=1,
        features=[64, 128, 256, 512], use_attention=True,
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    return model

def load_autoencoder(path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = Autoencoder(in_channels=4, features=[64, 128, 256, 512])
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    return model

model_sup = load_supervised('siamese_unet_best.pt', device)
model_ss = load_autoencoder('autoencoder_best.pt', device)

print("Models loaded")

## 2. Evaluate on Validation Set

In [ ]:
class OSCDWithNDVI(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image'].float()
        mask = sample['mask'].float()
        if image.dim() == 4:
            t1, t2 = image[0], image[1]
        else:
            t1 = t2 = image
        t1 = torch.cat([t1, compute_ndvi_torch(t1[0], t1[2]).unsqueeze(0)])
        t2 = torch.cat([t2, compute_ndvi_torch(t2[0], t2[2]).unsqueeze(0)])
        return {'image': torch.stack([t1, t2], dim=0), 'mask': mask.unsqueeze(0),
                't1': t1, 't2': t2}

root = os.path.join(tempfile.gettempdir(), 'oscd100')
val_dataset = OSCD100(root=root, split='val', bands=OSCD100.rgb_bands, download=True)
val_ds = OSCDWithNDVI(val_dataset)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)

print(f"Val samples: {len(val_ds)}")

In [ ]:
all_preds_sup = []
all_preds_ss = []
all_masks = []
all_t2 = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        images = batch['image'].to(device)
        masks = batch['mask']
        t1 = batch['t1'].to(device)
        t2 = batch['t2'].to(device)

        stacked = torch.cat([images[:, 0], images[:, 1]], dim=1)
        pred_sup = torch.sigmoid(model_sup(stacked))
        all_preds_sup.append(pred_sup.cpu())

        recon_t1 = model_ss(t1)
        recon_t2 = model_ss(t2)
        err_t1 = torch.mean((t1 - recon_t1)**2, dim=1, keepdim=True)
        err_t2 = torch.mean((t2 - recon_t2)**2, dim=1, keepdim=True)
        change_map = torch.abs(err_t1 - err_t2)
        all_preds_ss.append(change_map.cpu())

        all_masks.append(masks)
        all_t2.append(t2.cpu())

all_preds_sup = torch.cat(all_preds_sup)
all_preds_ss = torch.cat(all_preds_ss)
all_masks = torch.cat(all_masks)

print(f"Predictions: {all_preds_sup.shape}")

## 3. Compute Metrics Across Thresholds

In [ ]:
results = {}

for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    sup_metrics = evaluate_all(all_preds_sup, all_masks, threshold=thresh)
    results[f'Supervised@{thresh}'] = sup_metrics

for thresh in np.arange(0.05, 0.40, 0.05):
    ss_pred = (all_preds_ss > thresh).float()
    ss_metrics = evaluate_all(ss_pred, all_masks, threshold=0.5)
    results[f'Self-Supervised@{thresh:.2f}'] = ss_metrics

print(f"{'Method':<30} {'F1':>8} {'IoU':>8} {'Prec':>8} {'Recall':>8}")
print("-" * 62)
for name, m in sorted(results.items(), key=lambda x: -x[1]['f1']):
    print(f"{name:<30} {m['f1']:>8.4f} {m['iou']:>8.4f} {m['precision']:>8.4f} {m['recall']:>8.4f}")

## 4. Best Threshold Comparison

In [ ]:
best_sup_thresh = max([k for k in results if k.startswith('Supervised')],
                       key=lambda k: results[k]['f1'])
best_ss_thresh = max([k for k in results if k.startswith('Self-Supervised')],
                      key=lambda k: results[k]['f1'])

best_sup = results[best_sup_thresh]
best_ss = results[best_ss_thresh]

summary = {
    'Siamese U-Net (Supervised)': best_sup,
    'Autoencoder (Self-Supervised)': best_ss,
}

fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['f1', 'iou', 'precision', 'recall']
metric_labels = ['F1 Score', 'IoU', 'Precision', 'Recall']
x = np.arange(len(metrics))
width = 0.35

sup_vals = [best_sup[m] for m in metrics]
ss_vals = [best_ss[m] for m in metrics]

bars1 = ax.bar(x - width/2, sup_vals, width, label='Supervised', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, ss_vals, width, label='Self-Supervised', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Model Comparison: Best Threshold', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, axis='y', alpha=0.3)

for bar, val in zip(bars1, sup_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, ss_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Threshold Sensitivity Curves

In [ ]:
sup_f1s = []
sup_ious = []
sup_threshs = np.arange(0.1, 0.9, 0.05)
for t in sup_threshs:
    m = evaluate_all(all_preds_sup, all_masks, threshold=t)
    sup_f1s.append(m['f1'])
    sup_ious.append(m['iou'])

ss_f1s = []
ss_ious = []
ss_threshs = np.arange(0.02, 0.40, 0.02)
for t in ss_threshs:
    pred = (all_preds_ss > t).float()
    m = evaluate_all(pred, all_masks, threshold=0.5)
    ss_f1s.append(m['f1'])
    ss_ious.append(m['iou'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sup_threshs, sup_f1s, 'o-', linewidth=2, label='F1', color='#3498db')
axes[0].plot(sup_threshs, sup_ious, 's-', linewidth=2, label='IoU', color='#2ecc71')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Supervised: Threshold vs Metrics')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ss_threshs, ss_f1s, 'o-', linewidth=2, label='F1', color='#e74c3c')
axes[1].plot(ss_threshs, ss_ious, 's-', linewidth=2, label='IoU', color='#f39c12')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Self-Supervised: Threshold vs Metrics')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Threshold Sensitivity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Qualitative Results Grid

In [ ]:
n_show = min(6, len(val_ds))
fig, axes = plt.subplots(n_show, 5, figsize=(25, 5*n_show))

for i in range(n_show):
    sample = val_ds[i]
    images = sample['image'].unsqueeze(0).to(device)
    mask_gt = sample['mask'].squeeze().numpy()

    with torch.no_grad():
        stacked = torch.cat([images[:, 0], images[:, 1]], dim=1)
        pred_sup = torch.sigmoid(model_sup(stacked)).squeeze().cpu().numpy()

        t1_dev = sample['t1'].unsqueeze(0).to(device)
        t2_dev = sample['t2'].unsqueeze(0).to(device)
        recon_t1 = model_ss(t1_dev)
        recon_t2 = model_ss(t2_dev)
        err = torch.abs(
            torch.mean((t1_dev - recon_t1)**2, dim=1) -
            torch.mean((t2_dev - recon_t2)**2, dim=1)
        ).squeeze().cpu().numpy()

    t2_img = images[0, 1, :3].cpu().numpy().transpose(1, 2, 0)
    t2_img = np.clip(t2_img / (t2_img.max() + 1e-10), 0, 1)

    axes[i, 0].imshow(t2_img)
    axes[i, 0].set_title('After (T2)')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask_gt, cmap='gray')
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')

    axes[i, 2].imshow((pred_sup > 0.5).astype(np.uint8), cmap='gray')
    f1_s = compute_f1(torch.from_numpy(pred_sup).unsqueeze(0), torch.from_numpy(mask_gt).unsqueeze(0), 0.5)
    axes[i, 2].set_title(f'Supervised (F1: {f1_s:.3f})')
    axes[i, 2].axis('off')

    axes[i, 3].imshow(err, cmap='hot')
    axes[i, 3].set_title('Recon Error Map')
    axes[i, 3].axis('off')

    pred_ss = (err > 0.15).astype(np.uint8)
    f1_ss = compute_f1(torch.from_numpy(pred_ss).float().unsqueeze(0), torch.from_numpy(mask_gt).float().unsqueeze(0))
    axes[i, 4].imshow(pred_ss, cmap='gray')
    axes[i, 4].set_title(f'Self-Sup (F1: {f1_ss:.3f})')
    axes[i, 4].axis('off')

plt.suptitle('Qualitative Results: All Methods', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('qualitative_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Final Summary Table

In [ ]:
print("=" * 80)
print("FINAL RESULTS SUMMARY")
print("=" * 80)
print()
print(f"Dataset: OSCD100 ({len(val_ds)} validation samples)")
print(f"Input: RGB + NDVI (4 channels per date)")
print()
print(f"{'Model':<35} {'F1':>8} {'IoU':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 70)
print(f"{'Siamese U-Net (Supervised)':<35} {best_sup['f1']:>8.4f} {best_sup['iou']:>8.4f} {best_sup['precision']:>10.4f} {best_sup['recall']:>8.4f}")
print(f"{'Autoencoder (Self-Supervised)':<35} {best_ss['f1']:>8.4f} {best_ss['iou']:>8.4f} {best_ss['precision']:>10.4f} {best_ss['recall']:>8.4f}")
print()
print(f"Best supervised threshold: {best_sup_thresh}")
print(f"Best self-supervised threshold: {best_ss_thresh}")
print()
print("Interview narrative:")
print("  'I built a Siamese U-Net change detection model on Sentinel-2 imagery,")
print(f"   achieving F1 of {best_sup['f1']:.3f} on the OSCD benchmark. I also implemented")
print(f"   a self-supervised autoencoder variant (F1: {best_ss['f1']:.3f}) that detects")
print("   changes without any labels - useful when annotation is expensive.")
print("   The supervised model was then applied to Patna (Boring Road + Kurji)")
print("   to detect urban expansion between 2022-2024.")")
print()
print("Figures saved:")
print("  - model_comparison.png")
print("  - threshold_sensitivity.png")
print("  - qualitative_results.png")

## 8. Save Results JSON

In [ ]:
final_results = {
    'dataset': 'OSCD100',
    'n_val_samples': len(val_ds),
    'input_channels': 'RGB + NDVI (4 per date)',
    'models': {
        'supervised': {
            'architecture': 'Siamese U-Net',
            'best_threshold': float(best_sup_thresh.split('@')[1]),
            'metrics': {k: float(v) for k, v in best_sup.items()},
        },
        'self_supervised': {
            'architecture': 'Convolutional Autoencoder',
            'best_threshold': float(best_ss_thresh.split('@')[1]),
            'metrics': {k: float(v) for k, v in best_ss.items()},
        },
    },
    'patna_aoi': {
        'name': 'Patna - Boring Road & Kurji',
        'bounds': [85.08, 25.57, 85.12, 25.62],
    },
}

with open('final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Results saved to final_results.json")